# 02 · Data Cleaning
**Netflix Content Analytics — Phase 2**

In Phase 1 we built an **issue log** of 12 problems. This notebook works through that log, one
decision at a time. The rules we follow:

1. **Never touch the raw file.** We read it and write a *new* cleaned file to `data/processed/`.
2. **Every change is justified.** If we can't explain a change in one sentence, we don't make it.
3. **Don't invent data.** A missing value is labelled or left missing — never guessed.
4. **Validate at the end.** Automated checks (`assert`) prove the output matches our intentions.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW_DATA_PATH = Path("..") / "data" / "raw" / "netflix_titles.csv"
PROCESSED_DIR = Path("..") / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

netflix_raw = pd.read_csv(RAW_DATA_PATH)
netflix_clean = netflix_raw.copy()  # .copy() so edits never leak back into netflix_raw

ROWS_BEFORE = len(netflix_clean)
print(f"Starting rows: {ROWS_BEFORE:,}")

Starting rows: 8,807


## Step 1 · Clean hidden characters in text (Issue #9)
Non-breaking spaces (` `) and zero-width characters look like nothing on screen, but they make
`"Consequences"` and `"Consequences "` two different strings. We replace them with normal spaces and
trim leading/trailing whitespace (this also fixes the leading spaces in `date_added`, Issue #1).

In [2]:
HIDDEN_CHARS = {" ": " ", "​": "", "‌": "", "‍": "", "﻿": ""}

text_columns = netflix_clean.select_dtypes("str").columns
for column in text_columns:
    for hidden, replacement in HIDDEN_CHARS.items():
        netflix_clean[column] = netflix_clean[column].str.replace(hidden, replacement, regex=False)
    netflix_clean[column] = netflix_clean[column].str.strip()

changed_cells = (netflix_clean[text_columns].fillna("") != netflix_raw[text_columns].fillna("")).sum()
changed_cells[changed_cells > 0]

title          10
cast            1
date_added     88
description     9
dtype: int64

## Step 2 · Repair the shifted `rating` → `duration` rows (Issue #3)
For three stand-up specials the runtime (e.g. `"74 min"`) sits in `rating` and `duration` is empty.
The evidence is unambiguous (a rating can never be "74 min"), so we **move** the value back.
We do *not* guess the real rating — it becomes missing and will be labelled `Unknown` later.

In [3]:
shifted_rows = netflix_clean["rating"].str.contains(r"^\d+ min$", na=False) & netflix_clean["duration"].isna()
print("Rows to repair:", shifted_rows.sum())

netflix_clean.loc[shifted_rows, "duration"] = netflix_clean.loc[shifted_rows, "rating"]
netflix_clean.loc[shifted_rows, "rating"] = np.nan

netflix_clean.loc[shifted_rows, ["show_id", "title", "rating", "duration"]]

Rows to repair: 3


,show_id,title,rating,duration
5541,s5542,Louis C.K. 2017,NaN,74 min
5794,s5795,Louis C.K.: Hilarious,NaN,84 min
5813,s5814,Louis C.K.: Live at the Comedy Store,NaN,66 min


## Step 3 · Remove proven duplicates (Issue #10)
Now that hidden characters are gone, a true duplicate is a row identical to another in **every column
except `show_id`**. We print what we are about to drop *before* dropping it.

In [4]:
content_columns = [c for c in netflix_clean.columns if c != "show_id"]
is_duplicate = netflix_clean.duplicated(subset=content_columns, keep="first")

netflix_clean[netflix_clean.duplicated(subset=content_columns, keep=False)][["show_id", "title", "type", "release_year", "date_added"]]

,show_id,title,type,release_year,date_added
3371,s3372,Consequences,Movie,2014,"October 25, 2019"
6529,s6530,Consequences,Movie,2014,"October 25, 2019"


In [5]:
dropped_ids = netflix_clean.loc[is_duplicate, "show_id"].tolist()
netflix_clean = netflix_clean[~is_duplicate].reset_index(drop=True)
print(f"Dropped {len(dropped_ids)} duplicate(s): {dropped_ids}")

Dropped 1 duplicate(s): ['s6530']


🧠 The three *ambiguous* same-title pairs (e.g. *Love in a Puff*) differ in `date_added`, so this rule
correctly keeps them. We'll mention them as a limitation instead of deleting on a hunch.

## Step 4 · Convert `date_added` into a real date (Issues #1, #2, #12)
We pass an **explicit format** (`%B %d, %Y` = "September 25, 2021"). Explicit formats are faster and
fail loudly instead of silently guessing day/month order. Missing dates stay missing (`NaT`).

In [6]:
netflix_clean["date_added"] = pd.to_datetime(netflix_clean["date_added"], format="%B %d, %Y")

print("Missing dates:", netflix_clean["date_added"].isna().sum())
print("Range:", netflix_clean["date_added"].min().date(), "→", netflix_clean["date_added"].max().date())

Missing dates: 10
Range: 2008-01-01 → 2021-09-25


## Step 5 · Feature engineering — dates
| New column | Meaning | Why it's useful |
|---|---|---|
| `added_year` | Year the title was added | Yearly addition trends |
| `added_month` / `added_month_name` | Month added | Seasonality |
| `content_age_at_add` | `added_year − release_year` | Is Netflix adding new releases or older library titles? |

We use pandas' nullable integer type `Int64` (capital I) so that missing dates stay missing instead of
turning the whole column into floats like `2019.0`.

In [7]:
netflix_clean["added_year"] = netflix_clean["date_added"].dt.year.astype("Int64")
netflix_clean["added_month"] = netflix_clean["date_added"].dt.month.astype("Int64")
netflix_clean["added_month_name"] = netflix_clean["date_added"].dt.strftime("%b")
netflix_clean["content_age_at_add"] = netflix_clean["added_year"] - netflix_clean["release_year"]

netflix_clean["content_age_at_add"].describe().round(1)

count    8796.0
mean        4.7
std         8.8
min        -3.0
25%         0.0
50%         1.0
75%         5.0
max        93.0
Name: content_age_at_add, dtype: Float64

In [8]:
added_before_release = netflix_clean["content_age_at_add"] < 0
print("Titles whose add year is BEFORE release year:", added_before_release.sum())
netflix_clean.loc[added_before_release, ["title", "type", "release_year", "date_added"]].head()

Titles whose add year is BEFORE release year: 14


,title,type,release_year,date_added
1551,Hilda,TV Show,2021,2020-12-14
1696,Polly Pocket,TV Show,2021,2020-11-15
2920,Love Is Blind,TV Show,2021,2020-02-13
3168,Fuller House,TV Show,2020,2019-12-06
3287,Maradona in Mexico,TV Show,2020,2019-11-13


🔍 A small number of titles — mostly TV shows — were "added" before their release year. The likely reason:
for series, `release_year` may reflect the **latest season**, while `date_added` reflects when the **show**
first arrived. We keep these values (they're not provably wrong) but remember this when interpreting
`content_age_at_add` for TV shows.

## Step 6 · Split `duration` into value + unit (Issue #5)
`"90 min"` → `90` + `minutes`; `"2 Seasons"` → `2` + `seasons`. A regular expression captures the number
and the word separately. Mapping both "Season" and "Seasons" to `seasons` removes the singular/plural mess.

In [9]:
duration_parts = netflix_clean["duration"].str.extract(r"^(?P<value>\d+)\s+(?P<unit>\w+)$")

netflix_clean["duration_value"] = duration_parts["value"].astype("Int64")
netflix_clean["duration_unit"] = duration_parts["unit"].str.lower().map(
    {"min": "minutes", "season": "seasons", "seasons": "seasons"}
)

pd.crosstab(netflix_clean["type"], netflix_clean["duration_unit"])

duration_unit,minutes,seasons
type,,
Movie,6130,0
TV Show,0,2676


## Step 7 · Label missing categorical values (Issues #4, #6, #7)
Why labels instead of deleting rows? Deleting every row with a missing director would remove most TV shows
and badly bias the analysis. A label keeps the row and makes the gap **visible** in charts.

| Column | Label | Reasoning |
|---|---|---|
| `director`, `cast` | `Not Listed` | Missing is structural (esp. TV) — not "unknown person" |
| `country` | `Unknown` | Production country genuinely not recorded |
| `rating` | `Unknown` | Includes the 3 repaired rows — we won't guess a rating |
| `date_added` | *left missing* | Inventing a date would distort time trends |

In [10]:
fill_labels = {"director": "Not Listed", "cast": "Not Listed", "country": "Unknown", "rating": "Unknown"}
print("Values being labelled:", {col: int(netflix_clean[col].isna().sum()) for col in fill_labels})
netflix_clean = netflix_clean.fillna(fill_labels)

Values being labelled: {'director': 2634, 'cast': 825, 'country': 831, 'rating': 7}


## Step 8 · Helpful grouped columns
**Audience group.** There are many rating codes, mixing TV (TV-MA) and film (R) systems. Grouping them into
audience bands makes charts easier to read. This mapping is our *analytical choice*, based on the
published US guidelines — not a Netflix classification.

In [11]:
RATING_TO_AUDIENCE = {
    "TV-Y": "Kids", "TV-Y7": "Kids", "TV-Y7-FV": "Kids",
    "G": "Family", "TV-G": "Family", "PG": "Family", "TV-PG": "Family",
    "PG-13": "Teens", "TV-14": "Teens",
    "R": "Adults", "NC-17": "Adults", "TV-MA": "Adults",
    "NR": "Not Rated", "UR": "Not Rated", "Unknown": "Not Rated",
}
netflix_clean["audience_group"] = netflix_clean["rating"].map(RATING_TO_AUDIENCE)
assert netflix_clean["audience_group"].notna().all(), "A rating code is missing from the mapping"

netflix_clean.groupby(["audience_group", "rating"]).size().rename("titles").to_frame()

titles
audience_group rating          
Adults         NC-17          3
               R            799
               TV-MA       3206
Family         G             41
               PG           287
               TV-G         220
               TV-PG        863
Kids           TV-Y         307
               TV-Y7        334
               TV-Y7-FV       6
Not Rated      NR            80
               UR             3
               Unknown        7
Teens          PG-13        490
               TV-14       2160

**Primary country.** For quick single-country views we keep the *first* listed country.
⚠️ Assumption: the first country listed is the main producer. The dataset doesn't promise that, so the
full multi-country analysis uses the bridge table built next.

In [12]:
# First tidy the list itself: ", South Korea" or "Poland," → drop empty pieces left by stray commas
netflix_clean["country"] = netflix_clean["country"].str.split(",").apply(
    lambda parts: ", ".join(p.strip() for p in parts if p.strip())
)

netflix_clean["primary_country"] = netflix_clean["country"].str.split(", ").str[0]
netflix_clean["country_count"] = netflix_clean["country"].str.count(", ") + 1
netflix_clean.loc[netflix_clean["country"] == "Unknown", "country_count"] = 0

# Check the previously broken rows
netflix_clean.loc[netflix_clean["show_id"].isin(["s194", "s366", "s1193"]), ["show_id", "country", "primary_country", "country_count"]]

,show_id,country,primary_country,country_count
193,s194,South Korea,South Korea,1
365,s366,"France, Algeria",France,2
1192,s1193,United Kingdom,United Kingdom,1


## Step 9 · Normalise multi-value columns into bridge tables (Issue #8)
A cell like `"United States, India"` breaks counting. The fix is a **long / bridge table**:

| show_id | country |
|---|---|
| s7 | United States |
| s7 | India |

This is exactly how a relational database (and Power BI) wants many-to-many data.
Pandas makes it easy: `str.split` → `explode` → `strip` → drop empties (from stray commas).

In [13]:
title_countries = (
    netflix_clean[["show_id", "country"]]
    .assign(country=lambda df: df["country"].str.split(","))
    .explode("country")
    .assign(country=lambda df: df["country"].str.strip())
    .query("country != ''")
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"title_countries: {len(title_countries):,} rows | {title_countries['country'].nunique()} distinct values")
title_countries.head()

title_countries: 10,842 rows | 123 distinct values


,show_id,country
0,s1,United States
1,s2,South Africa
2,s3,Unknown
3,s4,Unknown
4,s5,India


**Genres.** Netflix categories are type-specific ("Dramas" vs "TV Dramas"). We keep the original label
**and** add a `genre_group` that merges movie/TV variants so we can compare across types. Labels that are not
really genres (e.g. "International Movies", "British TV Shows") are grouped separately so we can exclude them
from genre rankings when appropriate.

In [14]:
GENRE_GROUPS = {
    "International Movies": "International (label)", "International TV Shows": "International (label)",
    "British TV Shows": "Regional TV (label)", "Spanish-Language TV Shows": "Regional TV (label)",
    "Korean TV Shows": "Regional TV (label)",
    "Dramas": "Drama", "TV Dramas": "Drama",
    "Comedies": "Comedy", "TV Comedies": "Comedy",
    "Stand-Up Comedy": "Stand-Up & Talk", "Stand-Up Comedy & Talk Shows": "Stand-Up & Talk",
    "Documentaries": "Documentary", "Docuseries": "Documentary", "Science & Nature TV": "Documentary",
    "Action & Adventure": "Action & Adventure", "TV Action & Adventure": "Action & Adventure",
    "Independent Movies": "Independent",
    "Children & Family Movies": "Kids & Family", "Kids' TV": "Kids & Family",
    "Romantic Movies": "Romance", "Romantic TV Shows": "Romance",
    "Thrillers": "Thriller & Mystery", "TV Thrillers": "Thriller & Mystery", "TV Mysteries": "Thriller & Mystery",
    "Crime TV Shows": "Crime",
    "Horror Movies": "Horror", "TV Horror": "Horror",
    "Sci-Fi & Fantasy": "Sci-Fi & Fantasy", "TV Sci-Fi & Fantasy": "Sci-Fi & Fantasy",
    "Music & Musicals": "Music & Musicals",
    "Reality TV": "Reality",
    "Sports Movies": "Sports",
    "Anime Series": "Anime", "Anime Features": "Anime",
    "Classic Movies": "Classic & Cult", "Cult Movies": "Classic & Cult", "Classic & Cult TV": "Classic & Cult",
    "LGBTQ Movies": "LGBTQ",
    "Teen TV Shows": "Teen",
    "Faith & Spirituality": "Faith & Spirituality",
    "Movies": "Uncategorised", "TV Shows": "Uncategorised",
}

title_genres = (
    netflix_clean[["show_id", "listed_in"]]
    .assign(genre=lambda df: df["listed_in"].str.split(","))
    .explode("genre")
    .assign(genre=lambda df: df["genre"].str.strip())
    .query("genre != ''")
    [["show_id", "genre"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
title_genres["genre_group"] = title_genres["genre"].map(GENRE_GROUPS)
unmapped = title_genres.loc[title_genres["genre_group"].isna(), "genre"].unique()
assert len(unmapped) == 0, f"Unmapped genres: {unmapped}"

netflix_clean["genre_count"] = netflix_clean["show_id"].map(title_genres.groupby("show_id").size())
print(f"title_genres: {len(title_genres):,} rows | {title_genres['genre'].nunique()} labels → "
      f"{title_genres['genre_group'].nunique()} groups")
title_genres.head()

title_genres: 19,320 rows | 42 labels → 23 groups


,show_id,genre,genre_group
0,s1,Documentaries,Documentary
1,s2,International TV Shows,International (label)
2,s2,TV Dramas,Drama
3,s2,TV Mysteries,Thriller & Mystery
4,s3,Crime TV Shows,Crime


## Step 10 · Final column order

In [15]:
FINAL_COLUMNS = [
    "show_id", "type", "title", "director", "cast", "country", "primary_country", "country_count",
    "date_added", "added_year", "added_month", "added_month_name", "release_year", "content_age_at_add",
    "rating", "audience_group", "duration", "duration_value", "duration_unit",
    "listed_in", "genre_count", "description",
]
netflix_clean = netflix_clean[FINAL_COLUMNS]
netflix_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 8806 entries, 0 to 8805
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   show_id             8806 non-null   str           
 1   type                8806 non-null   str           
 2   title               8806 non-null   str           
 3   director            8806 non-null   str           
 4   cast                8806 non-null   str           
 5   country             8806 non-null   str           
 6   primary_country     8806 non-null   object        
 7   country_count       8806 non-null   int64         
 8   date_added          8796 non-null   datetime64[us]
 9   added_year          8796 non-null   Int64         
 10  added_month         8796 non-null   Int64         
 11  added_month_name    8796 non-null   str           
 12  release_year        8806 non-null   int64         
 13  content_age_at_add  8796 non-null   Int64         
 14  rat

## Step 11 · Validate the cleaned data ✅
`assert` statements are tiny automated tests. If any assumption is broken, the notebook stops with an
error instead of silently producing wrong results. This is a habit that impresses reviewers.

In [16]:
checks = {
    "Exactly one row removed (the proven duplicate)": len(netflix_clean) == ROWS_BEFORE - 1,
    "show_id is unique": netflix_clean["show_id"].is_unique,
    "type is only Movie / TV Show": set(netflix_clean["type"]) == {"Movie", "TV Show"},
    "No durations left in rating": not netflix_clean["rating"].str.contains("min").any(),
    "Every title has a duration value": netflix_clean["duration_value"].notna().all(),
    "Movies measured in minutes": (netflix_clean.query("type == 'Movie'")["duration_unit"] == "minutes").all(),
    "TV shows measured in seasons": (netflix_clean.query("type == 'TV Show'")["duration_unit"] == "seasons").all(),
    "date_added is a datetime": pd.api.types.is_datetime64_any_dtype(netflix_clean["date_added"]),
    "Only date_added-derived fields may be missing": set(netflix_clean.columns[netflix_clean.isna().any()])
        <= {"date_added", "added_year", "added_month", "added_month_name", "content_age_at_add"},
    "No hidden non-breaking spaces left": not netflix_clean["title"].str.contains(" ").any(),
    "No empty primary_country (stray commas fixed)": netflix_clean["primary_country"].str.len().gt(0).all(),
    "Every title has at least one genre":netflix_clean["genre_count"].ge(1).all(),
    "Bridge tables reference valid show_ids": set(title_countries["show_id"]) <= set(netflix_clean["show_id"])
        and set(title_genres["show_id"]) <= set(netflix_clean["show_id"]),
}
for name, passed in checks.items():
    print(("PASS " if passed else "FAIL ") + name)
assert all(checks.values()), "At least one validation check failed"

PASS Exactly one row removed (the proven duplicate)
PASS show_id is unique
PASS type is only Movie / TV Show
PASS No durations left in rating
PASS Every title has a duration value
PASS Movies measured in minutes
PASS TV shows measured in seasons
PASS date_added is a datetime
PASS Only date_added-derived fields may be missing
PASS No hidden non-breaking spaces left
PASS No empty primary_country (stray commas fixed)
PASS Every title has at least one genre
PASS Bridge tables reference valid show_ids


## Step 12 · Save the processed files
We save three CSVs — the main table plus two bridge tables. Dates are written in ISO format
(`2021-09-25`) so SQL and Power BI read them without guessing.

In [17]:
outputs = {
    "netflix_titles_clean.csv": netflix_clean,
    "netflix_title_countries.csv": title_countries,
    "netflix_title_genres.csv": title_genres,
}
for filename, frame in outputs.items():
    frame.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d", encoding="utf-8")
    print(f"Saved data/processed/{filename:<30} {len(frame):>6,} rows × {frame.shape[1]} cols")

Saved data/processed/netflix_titles_clean.csv        8,806 rows × 22 cols

Saved data/processed/netflix_title_countries.csv    10,842 rows × 2 cols
Saved data/processed/netflix_title_genres.csv       19,320 rows × 3 cols


## Cleaning summary

| Action | Rows affected |
|---|---|
| Hidden characters replaced / whitespace trimmed | see Step 1 table |
| Shifted `rating` → `duration` repaired | 3 |
| Proven duplicate removed | 1 (s6530) |
| Missing `director`/`cast` → `Not Listed`, `country`/`rating` → `Unknown` | see Step 7 |
| Missing `date_added` | kept missing |
| New features | `added_year`, `added_month`, `added_month_name`, `content_age_at_add`, `duration_value`, `duration_unit`, `audience_group`, `primary_country`, `country_count`, `genre_count` |
| New bridge tables | `netflix_title_countries.csv`, `netflix_title_genres.csv` |

✅ **Phase 2 complete.** The raw file is untouched; the cleaned data is validated. Next: `03_exploratory_analysis.ipynb`.